In [16]:
import os
import shutil
import random
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sys
!{sys.executable} -m pip install torch # I had an error with installing pytorch stack overflow said this would fix it
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn.functional as F 
from PIL import Image

# for kaggle hub
import sys
!{sys.executable} -m pip install kagglehub
import kagglehub

In [26]:
df = pd.read_csv("labels.csv")

group_map = {
    # 0: Speed limit
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 18: 0, 19: 0,

    # 1: Prohibited / No signs
    8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1,
    16: 1, 17: 1, 54: 1, 55: 1,

    # 2: Mandatory / Direction
    20: 2, 21: 2, 22: 2, 23: 2, 24: 2, 25: 2, 26: 2, 27: 2,
    29: 2, 31: 2, 43: 2, 44: 2, 53: 2,

    # 3: Warning / Hazard
    28: 3, 30: 3, 32: 3, 33: 3, 34: 3, 35: 3, 36: 3, 37: 3,
    38: 3, 39: 3, 46: 3, 47: 3, 48: 3, 50: 3, 51: 3
}

unknown_ids = [40, 41, 42, 45, 49, 52, 56, 57]

# Drop unknown data
df = df[~df["ClassId"].isin(unknown_ids)].copy()

# Create new 4-class label
df["GroupedClassId"] = df["ClassId"].map(group_map)

print("Missing grouped labels:", df["GroupedClassId"].isna().sum())
print(df["GroupedClassId"].value_counts().sort_index())

df.head()

Missing grouped labels: 0
GroupedClassId
0    10
1    12
2    13
3    15
Name: count, dtype: int64


,ClassId,Name,GroupedClassId
0,0,Speed limit (5km/h),0
1,1,Speed limit (15km/h),0
2,2,Speed limit (30km/h),0
3,3,Speed limit (40km/h),0
4,4,Speed limit (50km/h),0


In [27]:
path = kagglehub.dataset_download("ahemateja19bec1025/traffic-sign-dataset-classification")
print("Path to dataset files:", path)

input_test_dir = path + "/traffic_Data/TEST"
working_test_dir = path+ "traffic_Data/TEST"
working_validation_dir = path + "traffic_Data/VALIDATION"

Path to dataset files: /Users/thomas/.cache/kagglehub/datasets/ahemateja19bec1025/traffic-sign-dataset-classification/versions/2
